<a href="https://colab.research.google.com/github/AyazN/DLS-final-project/blob/fix/cross-encoder/notebooks/full_search_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/AyazN/DLS-final-project/blob/fix/cross-encoder/notebooks/full_search_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Model Search — Full Search Pipeline

This demonstration notebook connects the existing project artifacts into one end-to-end pipeline:

**processed metadata → precomputed embeddings → FAISS → BM25 + dense retrieval → RRF → cross-encoder → evaluation**

The notebook does not regenerate embeddings. It uses a low-memory mode by default and automatically caches the completed BM25 retriever on Google Drive before continuing. Colab High-RAM remains preferable for the full-quality configuration.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 1. Configuration

If the integration branch has already been merged, change `REPO_BRANCH` to `develop` or `main`.
The Drive configuration supports either an extracted encoder directory or the `.tar` archive produced by the earlier notebook.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/AyazN/DLS-final-project.git"
# Override with AISE_REPO_BRANCH after this branch is merged.
REPO_BRANCH = os.environ.get("AISE_REPO_BRANCH", "fix/cross-encoder")
PROJECT_ROOT = Path("/content/DLS-final-project")

DRIVE_ROOT = Path("/content/drive/MyDrive/AISE")

MODEL_DIR_NAME = "BAAI__bge-small-en-v1.5"
# To use MiniLM, switch to:
# MODEL_DIR_NAME = "BAAI__bge-small-en-v1.5"

DRIVE_EMBEDDING_CANDIDATES = [
    DRIVE_ROOT / "embeddings" / MODEL_DIR_NAME,
    DRIVE_ROOT / "data" / "processed" / "embeddings" / MODEL_DIR_NAME,
]

if MODEL_DIR_NAME == "sentence-transformers__all-MiniLM-L6-v2":
    DRIVE_EMBEDDING_ARCHIVE_CANDIDATES = [
        DRIVE_ROOT / "embeddings" / "minilm_embeddings_600k.tar",
        DRIVE_ROOT / "minilm_embeddings_600k.tar",
    ]

elif MODEL_DIR_NAME == "BAAI__bge-small-en-v1.5":
    DRIVE_EMBEDDING_ARCHIVE_CANDIDATES = [
        DRIVE_ROOT / "embeddings" / "bge_small_embeddings_600k.tar",
        DRIVE_ROOT / "bge_small_embeddings_600k.tar",
    ]

else:
    raise ValueError(f"Unknown MODEL_DIR_NAME: {MODEL_DIR_NAME}")

LOCAL_EMBEDDING_DIR = (
    PROJECT_ROOT / "data" / "processed" / "embeddings" / MODEL_DIR_NAME
)
LOCAL_CLEAN_DATASET = PROJECT_ROOT / "data" / "processed" / "clean_dataset.parquet"

CLEAN_DATASET_CANDIDATES = [
    DRIVE_ROOT / "data" / "processed" / "clean_dataset.parquet",
    DRIVE_ROOT / "data" / "clean_dataset.parquet",
    DRIVE_ROOT / "clean_dataset.parquet",
]

INDEX_FILENAME = (
    "minilm_flat_ip.faiss"
    if MODEL_DIR_NAME.startswith("sentence-transformers")
    else "bge_small_flat_ip.faiss"
)
LOCAL_INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / INDEX_FILENAME
DRIVE_INDEX_PATH = DRIVE_ROOT / "indices" / INDEX_FILENAME
DRIVE_INDEX_CANDIDATES = [
    DRIVE_INDEX_PATH,
    DRIVE_ROOT / "data" / "indexes" / INDEX_FILENAME,
    DRIVE_ROOT / "data" / "processed" / "indexes" / INDEX_FILENAME,
]

RELEVANCE_PATH = DRIVE_ROOT / "evaluation" / "relevance.csv"
REPO_RELEVANCE_PATH = PROJECT_ROOT / "data" / "evaluation" / "relevance.csv"
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/AISE_my_results")
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_HF_CACHE = DRIVE_ROOT / "models" / "huggingface"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L6-v2"

RETRIEVAL_TOP_K = 100
RERANK_TOP_K = 20
RERANK_TEXT_MAX_CHARS = 500
SHORT_BODY_TEXT_MAX_CHARS = 200

# Safe defaults for a free Colab runtime with about 12.7 GB system RAM.
LOW_MEMORY_MODE = True
BODY_MAX_CHARS_OVERRIDE = 300
BM25_CACHE_COMPRESSION = 3


## 2. Repository and dependencies


In [ ]:
import importlib
import os
import subprocess
import sys

if (PROJECT_ROOT / ".git").exists():
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "fetch", "origin",
            f"refs/heads/{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}",
            "--prune",
        ],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "checkout", "-B",
            REPO_BRANCH, f"origin/{REPO_BRANCH}",
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", REPO_BRANCH, "--single-branch",
            REPO_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

stale_aise_modules = [
    name for name in sys.modules if name == "aise" or name.startswith("aise.")
]
for name in stale_aise_modules:
    del sys.modules[name]
importlib.invalidate_caches()

repo_revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()

print("Repository:", PROJECT_ROOT)
print("Branch:", REPO_BRANCH)
print("Revision:", repo_revision)
print("Cleared cached AISE modules:", len(stale_aise_modules))


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import inspect
import platform
from pathlib import Path

import aise
import numpy as np
import pandas as pd
import torch

from aise.contracts import EvaluationExample, Query
from aise.evaluation import RetrievalEvaluator, load_relevance_csv
from aise.pipeline import SearchPipeline
from aise.retrieval import (
    BM25Retriever,
    CrossEncoderReranker,
    DenseRetriever,
    HybridRetriever,
    RankFusionReranker,
    ReciprocalRankFusion,
)
from aise.search_index import (
    FaissFlatIndex,
    load_embedding_artifacts,
    load_search_documents,
)

expected_source_root = (PROJECT_ROOT / "src").resolve()
loaded_aise_path = Path(aise.__file__).resolve()
if expected_source_root not in loaded_aise_path.parents:
    raise RuntimeError(
        f"Loaded aise from {loaded_aise_path}, expected {expected_source_root}"
    )

required_document_loader_parameters = {"max_body_chars", "include_metadata"}
available_document_loader_parameters = set(
    inspect.signature(load_search_documents).parameters
)
missing_parameters = (
    required_document_loader_parameters - available_document_loader_parameters
)
required_reranker_parameters = {"text_strategy", "max_text_chars"}
missing_parameters |= required_reranker_parameters - set(
    inspect.signature(CrossEncoderReranker).parameters
)
if missing_parameters or not hasattr(CrossEncoderReranker, "rank_all"):
    raise RuntimeError(
        "The checked-out repository revision is too old for this notebook. "
        "Set REPO_BRANCH to develop and rerun from the top. "
        f"Missing API parameters: {sorted(missing_parameters)}"
    )

reranker_source = Path(inspect.getsourcefile(CrossEncoderReranker)).resolve()
if expected_source_root not in reranker_source.parents:
    raise RuntimeError(
        f"Loaded reranker from {reranker_source}, expected {expected_source_root}"
    )

print("AISE imports and reranker API compatibility verified")
print("Reranker source:", reranker_source)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Torch device:", "cuda" if torch.cuda.is_available() else "cpu")


## 3. Prepare artifacts from Google Drive

The embedding directory must contain `embeddings.npy`, `ids.npy`, `metadata.parquet`, `run_config.json`, `progress.json`, and `text_representation_samples.json`.


In [ ]:
import shutil

REQUIRED_ARTIFACTS = (
    "embeddings.npy",
    "ids.npy",
    "metadata.parquet",
    "run_config.json",
    "progress.json",
    "text_representation_samples.json",
)


def has_artifacts(directory: Path) -> bool:
    return all((directory / name).exists() for name in REQUIRED_ARTIFACTS)


if not has_artifacts(LOCAL_EMBEDDING_DIR):
    LOCAL_EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
    source_embedding_dir = next(
        (path for path in DRIVE_EMBEDDING_CANDIDATES if has_artifacts(path)),
        None,
    )
    source_archive = next(
        (path for path in DRIVE_EMBEDDING_ARCHIVE_CANDIDATES if path.exists()),
        None,
    )

    if source_embedding_dir is not None:
        print("Copying embedding artifacts from:", source_embedding_dir)
        for name in REQUIRED_ARTIFACTS:
            shutil.copy2(
                source_embedding_dir / name,
                LOCAL_EMBEDDING_DIR / name,
            )
    elif source_archive is not None:
        print("Extracting embedding archive from:", source_archive)
        subprocess.run(
            [
                "tar",
                "-xf",
                str(source_archive),
                "-C",
                str(PROJECT_ROOT),
            ],
            check=True,
        )
    else:
        raise FileNotFoundError(
            "Embedding directory/archive was not found on Google Drive. "
            "Update DRIVE_EMBEDDING_CANDIDATES."
        )

if not LOCAL_CLEAN_DATASET.exists():
    source_dataset = next(
        (path for path in CLEAN_DATASET_CANDIDATES if path.exists()),
        None,
    )
    if source_dataset is None:
        raise FileNotFoundError(
            "clean_dataset.parquet was not found. Update CLEAN_DATASET_CANDIDATES."
        )
    LOCAL_CLEAN_DATASET.parent.mkdir(parents=True, exist_ok=True)
    print("Copying processed metadata from:", source_dataset)
    shutil.copy2(source_dataset, LOCAL_CLEAN_DATASET)

assert has_artifacts(LOCAL_EMBEDDING_DIR)
assert LOCAL_CLEAN_DATASET.exists()

print("Embedding directory:", LOCAL_EMBEDDING_DIR)
print("Processed metadata:", LOCAL_CLEAN_DATASET)


## 4. Load and validate embedding artifacts


In [ ]:
from aise.search_index import load_embedding_artifacts

artifacts = load_embedding_artifacts(
    LOCAL_EMBEDDING_DIR,
    mmap_embeddings=True,
    expected_dim=384,
)

print("Shape:", artifacts.embeddings.shape)
print("Dtype:", artifacts.embeddings.dtype)
print("Encoder:", artifacts.model_name)
print("Normalized:", artifacts.normalized)
print("Metadata columns:", artifacts.metadata.columns.tolist())

ENCODER_MODEL_NAME = artifacts.model_name
NORMALIZE_QUERY_EMBEDDINGS = artifacts.normalized
ARTIFACT_ROW_COUNT = artifacts.embeddings.shape[0]
ARTIFACT_DIM = artifacts.embeddings.shape[1]
BODY_MAX_CHARS = (
    BODY_MAX_CHARS_OVERRIDE
    if BODY_MAX_CHARS_OVERRIDE is not None
    else int(artifacts.run_config.get("max_model_card_chars", 2500))
)

print("Low-memory mode:", LOW_MEMORY_MODE)
print("Maximum body characters used by retrieval:", BODY_MAX_CHARS)

assert artifacts.embeddings.shape == (600_000, 384)
assert artifacts.embeddings.dtype == np.float32
assert artifacts.normalized is True


## 5. Build or restore the BM25 retriever

The BM25 cache is written to Google Drive before the notebook continues. If the runtime disconnects later, the next run restores BM25 instead of rebuilding it. Low-memory mode drops bulky duplicated artifact metadata, but retains pipeline_tag and SearchDocument.tags because reranking requires them.


In [ ]:
%%time

import gc
import joblib
import psutil

from aise.retrieval import BM25Retriever
from aise.search_index import load_search_documents

BM25_CACHE_FORMAT_VERSION = 2

BM25_CACHE_PATH = (
    DRIVE_ROOT
    / "indices"
    / f"bm25_600k_body{BODY_MAX_CHARS}_meta{BM25_CACHE_FORMAT_VERSION}.joblib"
)
LOCAL_BM25_CACHE_PATH = Path(
    f"/content/bm25_600k_body{BODY_MAX_CHARS}_meta{BM25_CACHE_FORMAT_VERSION}.joblib"
)
BM25_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)


def print_system_memory(label: str) -> None:
    memory = psutil.virtual_memory()
    print(
        f"{label}: used={memory.used / 1024**3:.1f} GB, "
        f"available={memory.available / 1024**3:.1f} GB"
    )


if BM25_CACHE_PATH.exists():
    print("Copying cached BM25 retriever from Drive:", BM25_CACHE_PATH)
    del artifacts
    gc.collect()
    shutil.copy2(BM25_CACHE_PATH, LOCAL_BM25_CACHE_PATH)
    bundle = joblib.load(LOCAL_BM25_CACHE_PATH)
    LOCAL_BM25_CACHE_PATH.unlink(missing_ok=True)

    if bundle.get("format_version") != BM25_CACHE_FORMAT_VERSION:
        raise ValueError("Cached BM25 format lacks reranking metadata")
    if bundle.get("rerank_metadata_fields") != ("pipeline_tag", "tags"):
        raise ValueError("Cached BM25 documents lack required reranking metadata")
    if bundle.get("document_count") != ARTIFACT_ROW_COUNT:
        raise ValueError("Cached BM25 document count does not match the artifacts")
    if bundle.get("body_max_chars") != BODY_MAX_CHARS:
        raise ValueError("Cached BM25 body limit does not match the configuration")

    bm25 = bundle["retriever"]
    documents = bm25.documents
    del bundle
    gc.collect()
else:
    print("No BM25 cache found; creating aligned search documents...")
    documents = load_search_documents(
        artifacts,
        processed_metadata_path=LOCAL_CLEAN_DATASET,
        max_body_chars=BODY_MAX_CHARS,
        include_metadata=not LOW_MEMORY_MODE,
    )
    assert len(documents) == ARTIFACT_ROW_COUNT

    print("Documents:", len(documents))
    print("Example:", documents[0].model_id, documents[0].title)
    print("Body preview:", documents[0].body[:300])

    del artifacts
    gc.collect()
    print_system_memory("Before BM25 build")

    print("Building BM25 over the complete document collection...")
    bm25 = BM25Retriever(documents)
    print("BM25 build completed")
    print_system_memory("After BM25 build")

    bundle = {
        "format_version": BM25_CACHE_FORMAT_VERSION,
        "retriever": bm25,
        "document_count": len(documents),
        "body_max_chars": BODY_MAX_CHARS,
        "low_memory_mode": LOW_MEMORY_MODE,
        "rerank_metadata_fields": ("pipeline_tag", "tags"),
    }

    print("Serializing BM25 cache locally...")
    joblib.dump(
        bundle,
        LOCAL_BM25_CACHE_PATH,
        compress=BM25_CACHE_COMPRESSION,
    )
    cache_size_gb = LOCAL_BM25_CACHE_PATH.stat().st_size / 1024**3
    print(f"Local BM25 cache size: {cache_size_gb:.2f} GB")

    temporary_drive_path = BM25_CACHE_PATH.with_suffix(".joblib.tmp")
    print("Copying BM25 cache to Google Drive...")
    shutil.copy2(LOCAL_BM25_CACHE_PATH, temporary_drive_path)
    temporary_drive_path.replace(BM25_CACHE_PATH)
    LOCAL_BM25_CACHE_PATH.unlink(missing_ok=True)

    print("BM25 cache saved successfully:", BM25_CACHE_PATH)
    del bundle
    gc.collect()

assert len(documents) == ARTIFACT_ROW_COUNT
assert all("pipeline_tag" in doc.metadata for doc in documents[:1000])
print("BM25 is ready for", len(documents), "documents")
print_system_memory("BM25 stage completed")


## 6. FAISS index

The vectors are already L2-normalized, so `IndexFlatIP` provides cosine-equivalent ranking. The notebook first tries to load a cached index from Drive. If no index exists, it builds one from the precomputed embeddings and saves it for future runs.


In [ ]:
from aise.search_index import FaissFlatIndex, load_embedding_artifacts

import gc

# Reload the lightweight artifact handles after the memory-intensive BM25 stage.
artifacts = load_embedding_artifacts(
    LOCAL_EMBEDDING_DIR,
    mmap_embeddings=True,
    expected_dim=ARTIFACT_DIM,
)
artifact_ids = artifacts.ids
dense_metadata = None if LOW_MEMORY_MODE else artifacts.metadata

LOCAL_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
DRIVE_INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)

source_index = next(
    (path for path in DRIVE_INDEX_CANDIDATES if path.exists()),
    None,
)
if not LOCAL_INDEX_PATH.exists() and source_index is not None:
    print("Copying FAISS index from:", source_index)
    shutil.copy2(source_index, LOCAL_INDEX_PATH)

vector_index = FaissFlatIndex(
    dim=ARTIFACT_DIM,
    metric="inner_product",
)

if LOCAL_INDEX_PATH.exists():
    print("Loading FAISS index:", LOCAL_INDEX_PATH)
    vector_index.load(LOCAL_INDEX_PATH)
else:
    print("Building FAISS index from precomputed embeddings...")
    vector_index.build(artifacts.embeddings)
    vector_index.save(LOCAL_INDEX_PATH)
    shutil.copy2(LOCAL_INDEX_PATH, DRIVE_INDEX_PATH)
    print("Saved index to Drive:", DRIVE_INDEX_PATH)

assert vector_index.ntotal == ARTIFACT_ROW_COUNT
assert vector_index.dim == ARTIFACT_DIM

print("Vectors indexed:", vector_index.ntotal)
print("Metric:", vector_index.metric)
print("Higher is better:", vector_index.higher_is_better)

del artifacts
gc.collect()
print_system_memory("FAISS stage completed")


## 7. Encoder, dense retrieval, and RRF fusion


In [ ]:
import torch

import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

from sentence_transformers import SentenceTransformer

DRIVE_HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["SENTENCE_TRANSFORMERS_HOME"] = str(DRIVE_HF_CACHE)
encoder_device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = SentenceTransformer(
    ENCODER_MODEL_NAME,
    device=encoder_device,
    cache_folder=str(DRIVE_HF_CACHE),
)

print("Loaded encoder:", ENCODER_MODEL_NAME)
print("Encoder device:", encoder_device)
print("Persistent Hugging Face cache:", DRIVE_HF_CACHE)


In [ ]:
from aise.retrieval import (
    DenseRetriever,
    HybridRetriever,
    ReciprocalRankFusion,
)

dense = DenseRetriever(
    index=vector_index,
    documents=documents,
    model=encoder,
    encoder_name=ENCODER_MODEL_NAME,
    ids=artifact_ids,
    metadata=dense_metadata,
    normalize_embeddings=NORMALIZE_QUERY_EMBEDDINGS,
)

hybrid = HybridRetriever(
    bm25=bm25,
    dense=dense,
    fusion=ReciprocalRankFusion(k=60),
)


In [ ]:
import pandas as pd

from aise.contracts import Query


def results_frame(results, *, score_name: str = "score"):
    """Compact presentation table without duplicate identity columns."""
    return pd.DataFrame(
        [
            {
                "rank": result.rank,
                "model_id": result.model_id,
                score_name: result.score,
                "snippet": " ".join(result.snippet.split())[:220],
            }
            for result in results
        ]
    )


def overlap_at_k(left, right, k: int = 20) -> tuple[int, float]:
    left_ids = {result.doc_id for result in left[:k]}
    right_ids = {result.doc_id for result in right[:k]}
    intersection = len(left_ids & right_ids)
    union = len(left_ids | right_ids)
    return intersection, intersection / union if union else 0.0


DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
demo_query = Query(
    text="basketball video analysis model",
    top_k=RETRIEVAL_TOP_K,
)

bm25_demo_results = list(bm25.search(demo_query))
dense_demo_results = list(dense.search(demo_query))
hybrid_results = list(hybrid.search(demo_query))

result_tables = {
    "bm25_demo_results.csv": results_frame(bm25_demo_results, score_name="bm25_score"),
    "dense_demo_results.csv": results_frame(dense_demo_results, score_name="cosine_score"),
    "hybrid_rrf_demo_results.csv": results_frame(hybrid_results, score_name="rrf_score"),
}
for filename, table in result_tables.items():
    table.to_csv(DRIVE_RESULTS_DIR / filename, index=False)

top_results_by_system = pd.concat(
    [
        results_frame(bm25_demo_results[:5]).assign(system="BM25"),
        results_frame(dense_demo_results[:5]).assign(system="Dense / FAISS"),
        results_frame(hybrid_results[:5]).assign(system="Hybrid RRF"),
    ],
    ignore_index=True,
)[["system", "rank", "model_id", "score", "snippet"]]

bm25_by_id = {result.doc_id: result for result in bm25_demo_results}
dense_by_id = {result.doc_id: result for result in dense_demo_results}
rank_comparison = pd.DataFrame(
    [
        {
            "model_id": result.model_id,
            "bm25_rank": bm25_by_id.get(result.doc_id).rank if result.doc_id in bm25_by_id else None,
            "dense_rank": dense_by_id.get(result.doc_id).rank if result.doc_id in dense_by_id else None,
            "rrf_rank": result.rank,
            "bm25_score": bm25_by_id.get(result.doc_id).score if result.doc_id in bm25_by_id else None,
            "cosine_score": dense_by_id.get(result.doc_id).score if result.doc_id in dense_by_id else None,
            "rrf_score": result.score,
        }
        for result in hybrid_results[:20]
    ]
)
rank_comparison.to_csv(DRIVE_RESULTS_DIR / "retrieval_rank_comparison.csv", index=False)

overlap_rows = []
for left_name, left_results, right_name, right_results in (
    ("BM25", bm25_demo_results, "Dense", dense_demo_results),
    ("BM25", bm25_demo_results, "Hybrid RRF", hybrid_results),
    ("Dense", dense_demo_results, "Hybrid RRF", hybrid_results),
):
    overlap_count, jaccard = overlap_at_k(left_results, right_results, k=20)
    overlap_rows.append(
        {
            "rankings": f"{left_name} vs {right_name}",
            "shared_documents@20": overlap_count,
            "jaccard@20": jaccard,
        }
    )
overlap_table = pd.DataFrame(overlap_rows)
overlap_table.to_csv(DRIVE_RESULTS_DIR / "retrieval_overlap_comparison.csv", index=False)

print("Query:", demo_query.text)
print("Top five results from each retrieval stage:")
display(top_results_by_system)
print("How BM25 and dense ranks contribute to the final RRF top 20:")
display(rank_comparison)
print("Top-20 overlap between rankings:")
display(overlap_table)
print(
    "BM25, cosine, and RRF scores have different scales. "
    "Compare ranks, not raw scores."
)


## 8. Cross-encoder reranking


In [ ]:
from sentence_transformers import CrossEncoder

from aise.retrieval import CrossEncoderReranker

cross_encoder = CrossEncoder(
    CROSS_ENCODER_NAME,
    device=encoder_device,
)
reranker = CrossEncoderReranker(
    cross_encoder,
    top_k=RERANK_TOP_K,
    text_strategy="title_task_body",
    max_text_chars=RERANK_TEXT_MAX_CHARS,
)

missing_payload_fields = [
    result.doc_id
    for result in hybrid_results
    if "pipeline_tag" not in result.metadata or "tags" not in result.metadata
]
if missing_payload_fields:
    raise RuntimeError(
        "Hybrid candidates lack reranking metadata. Rebuild the versioned BM25 cache."
    )

payload_preview = pd.DataFrame(
    [
        {
            "model_id": result.model_id,
            "title": result.title,
            "pipeline_tag": result.metadata.get("pipeline_tag", ""),
            "tags": list(result.metadata.get("tags", ()))[:5],
            "body": result.metadata.get("body", "")[:160],
            "cross_encoder_text": reranker._document_text(result)[:240],
        }
        for result in hybrid_results[:5]
    ]
)
print("Exact payload fields passed to CrossEncoder.predict():")
display(payload_preview)

reranked_results = list(reranker.rank(demo_query, hybrid_results))
reranked_table = results_frame(reranked_results, score_name="cross_encoder_score")
reranked_table.to_csv(
    DRIVE_RESULTS_DIR / "cross_encoder_reranked_demo_results.csv",
    index=False,
)

hybrid_by_id = {result.doc_id: result for result in hybrid_results}
reranker_comparison = pd.DataFrame(
    [
        {
            "model_id": result.model_id,
            "hybrid_rank": hybrid_by_id[result.doc_id].rank,
            "reranked_rank": result.rank,
            "rank_change": hybrid_by_id[result.doc_id].rank - result.rank,
            "rrf_score": hybrid_by_id[result.doc_id].score,
            "cross_encoder_score": result.score,
        }
        for result in reranked_results
    ]
)
reranker_comparison.to_csv(
    DRIVE_RESULTS_DIR / "hybrid_vs_reranker_comparison.csv",
    index=False,
)

print("Query:", demo_query.text)
print(
    "Cross-encoder effect: positive rank_change means the model moved upward; "
    "negative means it moved downward."
)
display(reranker_comparison)


In [ ]:
from aise.contracts import Query
from aise.pipeline import SearchPipeline

search_pipeline = SearchPipeline(
    retriever=hybrid,
    ranker=reranker,
)


def search_models(text: str, candidate_k: int = RETRIEVAL_TOP_K):
    query = Query(text=text, top_k=candidate_k)
    return list(search_pipeline.search(query))


pipeline_demo_query = "small multilingual text classification model"
pipeline_demo_results = search_models(pipeline_demo_query)
pipeline_demo_table = results_frame(
    pipeline_demo_results,
    score_name="cross_encoder_score",
)
pipeline_demo_table.to_csv(
    DRIVE_RESULTS_DIR / "end_to_end_pipeline_demo_results.csv",
    index=False,
)

print("End-to-end query:", pipeline_demo_query)
print("Final top 20 after BM25 + dense + RRF + cross-encoder:")
display(pipeline_demo_table)


## 9. Evaluation

Evaluation uses the contract-compatible integration of `participants/05_evaluation`
exposed as `aise.evaluation.RetrievalEvaluator`. It reports Precision@K, Recall@K,
MRR, nDCG@K, and mean latency for BM25, dense, hybrid RRF, and the complete
cross-encoder pipeline.

The repository includes `data/evaluation/relevance.csv`, a reproducible **metadata-derived
benchmark** generated from `clean_dataset.parquet`. A query such as `video classification`
is relevant to every model whose normalized `pipeline_tag` exactly matches that task. The
file contains 10 queries and is validated against the searchable model IDs before evaluation.
It is independent of retrieval output, but it should be described as silver metadata labels,
not human expert judgments.

An optional `evaluation/relevance.csv` on Drive takes priority, so manually reviewed qrels
can replace the bundled benchmark without changing the notebook. If neither file exists,
the notebook derives a smaller fallback benchmark from artifact metadata.


In [ ]:
import gc
import json

from aise.contracts import EvaluationExample, Query
from aise.evaluation import load_relevance_csv


qrels_candidates = [
    (RELEVANCE_PATH, "external_relevance_csv", "manual qrels"),
    (
        REPO_RELEVANCE_PATH,
        "repository_metadata_qrels",
        "normalized pipeline_tag exact match",
    ),
]
selected_qrels = next(
    (
        (path, source, default_rule)
        for path, source, default_rule in qrels_candidates
        if path.exists()
    ),
    None,
)

if selected_qrels is not None:
    qrels_path, evaluation_source, default_label_rule = selected_qrels
    evaluation_examples = load_relevance_csv(
        qrels_path,
        top_k=RETRIEVAL_TOP_K,
    )
    relevance = pd.read_csv(qrels_path)
    evaluation_query_rows = []
    for example, (_, row) in zip(evaluation_examples, relevance.iterrows()):
        row_rule = row.get("label_rule", default_label_rule)
        label_rule = (
            str(row_rule).strip()
            if pd.notna(row_rule) and str(row_rule).strip()
            else default_label_rule
        )
        evaluation_query_rows.append(
            {
                "query": example.query.text,
                "label_rule": label_rule,
                "relevant_models": len(set(example.relevant_model_ids)),
            }
        )
    print("Using evaluation CSV:", qrels_path)
else:
    evaluation_metadata = pd.read_parquet(
        LOCAL_EMBEDDING_DIR / "metadata.parquet",
        columns=["model_id", "pipeline_tag"],
    )
    evaluation_metadata["model_id"] = evaluation_metadata["model_id"].astype(str)
    normalized_tags = (
        evaluation_metadata["pipeline_tag"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace("_", "-", regex=False)
        .str.replace(" ", "-", regex=False)
    )
    evaluation_metadata = evaluation_metadata.assign(normalized_pipeline_tag=normalized_tags)
    evaluation_metadata = evaluation_metadata[
        evaluation_metadata["normalized_pipeline_tag"] != ""
    ]

    tag_counts = evaluation_metadata["normalized_pipeline_tag"].value_counts()
    bounded_tags = tag_counts[(tag_counts >= 10) & (tag_counts <= 250)]
    candidate_tags = bounded_tags if len(bounded_tags) >= 5 else tag_counts[tag_counts >= 5]
    selected_tags = (
        candidate_tags.rename("relevant_models")
        .to_frame()
        .assign(distance_from_target=lambda frame: (frame["relevant_models"] - 50).abs())
        .sort_values(["distance_from_target", "relevant_models"], ascending=[True, False])
        .head(5)
    )
    if len(selected_tags) < 5:
        raise ValueError("Not enough populated pipeline_tag values for evaluation")

    evaluation_examples = []
    evaluation_query_rows = []
    for pipeline_tag, row in selected_tags.iterrows():
        relevant_ids = tuple(
            evaluation_metadata.loc[
                evaluation_metadata["normalized_pipeline_tag"] == pipeline_tag,
                "model_id",
            ].drop_duplicates()
        )
        query_text = pipeline_tag.replace("-", " ")
        evaluation_examples.append(
            EvaluationExample(
                query=Query(query_text, top_k=RETRIEVAL_TOP_K),
                relevant_model_ids=relevant_ids,
            )
        )
        evaluation_query_rows.append(
            {
                "query": query_text,
                "label_rule": f"pipeline_tag={pipeline_tag}",
                "relevant_models": len(relevant_ids),
            }
        )

    evaluation_source = "metadata_pipeline_tag_fallback"
    del evaluation_metadata, normalized_tags, tag_counts, bounded_tags, candidate_tags
    gc.collect()
    print(
        "No evaluation CSV found; using pipeline_tag metadata from the artifacts."
    )

required_evaluation_ids = {
    model_id
    for example in evaluation_examples
    for model_id in example.relevant_model_ids
}
found_evaluation_ids = {
    document.model_id
    for document in documents
    if document.model_id in required_evaluation_ids
}
missing_evaluation_ids = required_evaluation_ids - found_evaluation_ids
if missing_evaluation_ids:
    raise ValueError(
        "Evaluation relevance IDs are missing from the search documents: "
        f"{sorted(missing_evaluation_ids)[:20]}"
    )

evaluation_query_table = pd.DataFrame(evaluation_query_rows)
evaluation_query_table.to_csv(
    DRIVE_RESULTS_DIR / "evaluation_query_summary.csv",
    index=False,
)
print("Evaluation source:", evaluation_source)
display(evaluation_query_table)


In [ ]:
from aise.evaluation import RetrievalEvaluator

systems = {
    "bm25": bm25,
    "dense": dense,
    "hybrid_rrf": hybrid,
    "hybrid_rrf_cross_encoder": search_pipeline,
}

reports = {}
for system_name, system in systems.items():
    evaluator = RetrievalEvaluator(
        metric_k_values=(1, 5, 10, 20),
        top_k=RETRIEVAL_TOP_K,
    )
    reports[system_name] = evaluator.evaluate(evaluation_examples, system)

metrics_table = pd.DataFrame(
    {name: dict(report.metrics) for name, report in reports.items()}
).T
metrics_table.index.name = "system"
metrics_csv_path = DRIVE_RESULTS_DIR / "full_search_pipeline_metrics.csv"
metrics_table.to_csv(metrics_csv_path)

compact_metrics = (
    metrics_table[["mrr", "precision@5", "recall@20", "ndcg@10", "mean_latency_ms"]]
    .rename(
        columns={
            "mrr": "MRR",
            "precision@5": "Precision@5",
            "recall@20": "Recall@20",
            "ndcg@10": "nDCG@10",
            "mean_latency_ms": "Latency (ms)",
        }
    )
    .reset_index()
)

print("Evaluation source:", evaluation_source)
print("Higher is better for ranking metrics; lower is better for latency.")
display(compact_metrics.round(4))
print("Saved the complete metric set to:", metrics_csv_path)


## 10. Controlled reranking ablations

Vary only CrossEncoder fields and candidate-pool size, then compare pure CrossEncoder order with equal-weight Hybrid + CrossEncoder RRF.


In [ ]:
from aise.pipeline import SearchPipeline
from aise.retrieval import CrossEncoderReranker, RankFusionReranker

TEXT_VARIANTS = {
    "title_body": {
        "text_strategy": "title_body",
        "max_text_chars": RERANK_TEXT_MAX_CHARS,
    },
    "body_only": {
        "text_strategy": "body_only",
        "max_text_chars": RERANK_TEXT_MAX_CHARS,
    },
    "title_short_body": {
        "text_strategy": "title_body",
        "max_text_chars": SHORT_BODY_TEXT_MAX_CHARS,
    },
    "title_body_pipeline_tag": {
        "text_strategy": "title_task_body",
        "max_text_chars": RERANK_TEXT_MAX_CHARS,
    },
    "title_body_pipeline_tag_tags": {
        "text_strategy": "title_task_tags_body",
        "max_text_chars": RERANK_TEXT_MAX_CHARS,
    },
}
CANDIDATE_SIZES = (50, 100, 200)


def compact_report_row(name, report, **extra):
    metrics = report.metrics
    return {
        "system": name,
        **extra,
        "MRR": metrics["mrr"],
        "Precision@5": metrics["precision@5"],
        "Recall@20": metrics["recall@20"],
        "nDCG@10": metrics["ndcg@10"],
        "Latency (ms)": metrics["mean_latency_ms"],
    }


ablation_rows = []
ablation_reports = {}
for candidate_k in CANDIDATE_SIZES:
    for variant_name, variant_kwargs in TEXT_VARIANTS.items():
        experiment_reranker = CrossEncoderReranker(
            cross_encoder,
            top_k=RERANK_TOP_K,
            **variant_kwargs,
        )
        experiment_pipeline = SearchPipeline(hybrid, experiment_reranker)
        experiment_report = RetrievalEvaluator(
            metric_k_values=(1, 5, 10, 20),
            top_k=candidate_k,
        ).evaluate(evaluation_examples, experiment_pipeline)
        ablation_reports[(variant_name, candidate_k)] = experiment_report
        ablation_rows.append(
            compact_report_row(
                "Pure CrossEncoder",
                experiment_report,
                text_variant=variant_name,
                candidate_k=candidate_k,
            )
        )

ablation_table = pd.DataFrame(ablation_rows).sort_values(
    ["nDCG@10", "MRR"], ascending=False
)
ablation_table.to_csv(
    DRIVE_RESULTS_DIR / "reranker_ablation_metrics.csv", index=False
)
print("Field and candidate-size ablations:")
display(ablation_table.round(4))

best_ablation = ablation_table.iloc[0]
best_variant = str(best_ablation["text_variant"])
best_candidate_k = int(best_ablation["candidate_k"])
best_kwargs = TEXT_VARIANTS[best_variant]

fusion_pipeline = SearchPipeline(
    retriever=hybrid,
    ranker=RankFusionReranker(
        CrossEncoderReranker(
            cross_encoder,
            top_k=RERANK_TOP_K,
            **best_kwargs,
        ),
        cross_encoder_weight=1.0,
        k=60,
        final_k=RERANK_TOP_K,
    ),
)
fusion_evaluator = RetrievalEvaluator(
    metric_k_values=(1, 5, 10, 20),
    top_k=best_candidate_k,
)
fusion_report = fusion_evaluator.evaluate(evaluation_examples, fusion_pipeline)
hybrid_at_best_k = fusion_evaluator.evaluate(evaluation_examples, hybrid)
pure_ce_at_best_k = ablation_reports[(best_variant, best_candidate_k)]

fusion_table = pd.DataFrame(
    [
        compact_report_row(
            "Hybrid", hybrid_at_best_k,
            text_variant="n/a", candidate_k=best_candidate_k,
        ),
        compact_report_row(
            "Pure CrossEncoder", pure_ce_at_best_k,
            text_variant=best_variant, candidate_k=best_candidate_k,
        ),
        compact_report_row(
            "Hybrid + CE Fusion", fusion_report,
            text_variant=best_variant, candidate_k=best_candidate_k,
        ),
    ]
)
fusion_table.to_csv(
    DRIVE_RESULTS_DIR / "reranker_fusion_metrics.csv", index=False
)
print("Best pure-CE variant and equal-weight rank fusion:")
display(fusion_table.round(4))


In [ ]:
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

report_path = DRIVE_RESULTS_DIR / "full_search_pipeline_metrics.json"
with report_path.open("w", encoding="utf-8") as stream:
    json.dump(
        {
            "evaluation_source": evaluation_source,
            "systems": {
                name: {
                    "metrics": dict(report.metrics),
                    "details": dict(report.details),
                }
                for name, report in reports.items()
            },
        },
        stream,
        ensure_ascii=False,
        indent=2,
    )
print("Saved evaluation report:", report_path)

run_summary = {
    "status": "complete",
    "evaluation_status": "complete",
    "evaluation_source": evaluation_source,
    "encoder_model": ENCODER_MODEL_NAME,
    "cross_encoder_model": CROSS_ENCODER_NAME,
    "document_count": len(documents),
    "embedding_dimension": ARTIFACT_DIM,
    "body_max_chars": BODY_MAX_CHARS,
    "low_memory_mode": LOW_MEMORY_MODE,
    "retrieval_top_k": RETRIEVAL_TOP_K,
    "rerank_top_k": RERANK_TOP_K,
    "rerank_text_strategy": reranker.text_strategy,
    "rerank_text_max_chars": reranker.max_text_chars,
    "bm25_cache_format_version": BM25_CACHE_FORMAT_VERSION,
    "bm25_cache": str(BM25_CACHE_PATH),
    "faiss_index": str(DRIVE_INDEX_PATH),
    "evaluation_queries": len(evaluation_examples),
    "saved_result_files": sorted(
        path.name for path in DRIVE_RESULTS_DIR.glob("*.csv")
    ),
}
summary_path = DRIVE_RESULTS_DIR / "full_search_pipeline_run_summary.json"
with summary_path.open("w", encoding="utf-8") as stream:
    json.dump(run_summary, stream, ensure_ascii=False, indent=2)

print("Saved run summary:", summary_path)
print("The complete pipeline run, including evaluation, finished successfully")


## Ready for demonstration

The notebook persists all expensive reusable artifacts and presentation outputs:

- BM25 cache and FAISS index;
- Hugging Face encoder and cross-encoder cache;
- separate BM25, dense, hybrid, and reranked result CSV files;
- rank-change and top-20 overlap comparison tables;
- evaluation queries, label rules, and relevance counts;
- complete Precision, Recall, MRR, nDCG, and latency metrics;
- field/candidate-size ablations and Hybrid + CE fusion metrics;
- a final run summary written only after the complete pipeline succeeds.

`repository_metadata_qrels` means the checked-in, reproducible metadata-derived CSV was used.
`external_relevance_csv` means a Drive qrels file took priority.
`metadata_pipeline_tag_fallback` is used only if both CSV files are absent.
